# STAC Catalog Builder — Country boundaries (GeoParquet)

This notebook generates a STAC catalog for the preprocessed Natural Earth country boundaries GeoParquet and uploads it to Google Cloud.
It follows the vector pattern of `18_hydrolakes.ipynb` (parquet asset, `table:*` fields): one collection, one item. There is **no WMS/WFS asset and no GeoServer step**: the GeoServer of the atlas has no GeoParquet data store.

## Pipeline overview

| Step | Purpose |
|------|---------|
| 1. Configure paths and options | Input GeoParquet + metadata on `P:`, cloud targets (`gca-stac-7`) |
| 2. Define STAC helper functions | Collection, item and table-column builders |
| 3. Create the STAC collection | From `metadata_country_boundaries.json` |
| 4. Build the STAC item | One item for `country_boundaries.parquet` (`table:columns`, `table:row_count`) |
| 5. Save STAC catalog locally | `STAC/data/current/country_boundaries/` + root `catalog.json` |
| 6. Upload GeoParquet to Google Cloud | `gs://gca-data-public/gca/country_boundaries/country_boundaries.parquet` |
| 7. Upload STAC catalog to Google Cloud | `gs://gca-data-public/gca/gca-stac-7/` |
| 8. Verify the cloud copy | Public HTTPS href reachable, same size and row count as the local file |

**Prerequisite:** run `notebooks/19_country_boundaries.ipynb` first (via `docker/run_notebook_with_p_drive.ps1`) to produce `parquet/country_boundaries.parquet` and `metadata_country_boundaries.json` in `P:\...\Data_Vivian\stac\country_boundaries`.

Steps 6-7 are guarded by `RUN_UPLOAD`.

In [1]:
import datetime
import json
import os
from pathlib import Path
from posixpath import join as urljoin

import pyarrow.parquet as pq
import pystac
import requests
import shapely.geometry
from pystac.stac_io import DefaultStacIO

from coclicodata.etl.cloud_utils import load_google_credentials

PARQUET_MEDIA_TYPE = "application/vnd.apache.parquet"
PROJ_SCHEMA_URI = "https://stac-extensions.github.io/projection/v1.1.0/schema.json"
TABLE_SCHEMA_URI = "https://stac-extensions.github.io/table/v1.2.0/schema.json"

## 1) Configure paths and options

In [2]:
stac_data_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))
stac_dir = stac_data_repo / "current"

data_dir = Path(r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\country_boundaries")
parquet_path = data_dir / "parquet" / "country_boundaries.parquet"
metadata_path = data_dir / "metadata_country_boundaries.json"
google_cred_path = Path(
    r"P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\google_credentials.json"
)

RUN_UPLOAD = True  # set False to only build the STAC locally
gcs_protocol = "https://storage.googleapis.com"
gcs_project = "GCA - 11210264"
bucket_name = "gca-data-public"
bucket_proj = "gca"
stac_cloud_name = "gca-stac-7"

for p in (parquet_path, metadata_path):
    if not p.is_file():
        raise FileNotFoundError(f"{p} not found. Run notebooks/19_country_boundaries.ipynb first.")

with open(metadata_path, encoding="utf-8") as f:
    metadata = json.load(f)

REQUIRED_METADATA_KEYS = [
    "CRS", "COLLECTION_ID", "SPATIAL_EXTENT", "TEMPORAL_EXTENT", "TITLE", "TITLE_ABBREVIATION", "DESCRIPTION",
    "SHORT_DESCRIPTION", "LICENSE", "PROVIDERS", "KEYWORDS", "CITATION", "TABLE_COLUMNS",
]
missing_keys = [key for key in REQUIRED_METADATA_KEYS if key not in metadata]
if missing_keys:
    raise KeyError(f"Missing required metadata keys in {metadata_path}: {', '.join(missing_keys)}")
if None in metadata["SPATIAL_EXTENT"]:
    raise ValueError("SPATIAL_EXTENT is incomplete: re-run notebooks/19_country_boundaries.ipynb")

collection_id = metadata["COLLECTION_ID"]
proj_name = collection_id
href_prefix = urljoin(gcs_protocol, bucket_name, bucket_proj, proj_name)

print("GeoParquet input:", parquet_path)
print("STAC output:", stac_dir / collection_id)
print("Collection id:", collection_id)
print("Spatial extent:", metadata["SPATIAL_EXTENT"])
print("Cloud parquet href:", urljoin(href_prefix, parquet_path.name))

GeoParquet input: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\country_boundaries\parquet\country_boundaries.parquet
STAC output: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\country_boundaries
Collection id: country_boundaries
Spatial extent: [-180.0, -90.0, 180.0, 83.634101]
Cloud parquet href: https://storage.googleapis.com/gca-data-public/gca/country_boundaries/country_boundaries.parquet


## 2) Define STAC helper functions

In [3]:
def parse_license(metadata: dict) -> str:
    license_ = metadata["LICENSE"]
    if "Creative Commons" in license_ and "4.0" in license_:
        suffix = "-SA" if "ShareAlike" in license_ else ""
        return f"CC-BY{suffix}-4.0"
    return license_


def start_datetime(metadata: dict) -> datetime.datetime:
    start = metadata["TEMPORAL_EXTENT"][0].split("T")[0]
    return datetime.datetime.strptime(start, "%Y-%m-%d").replace(tzinfo=datetime.timezone.utc)


def table_columns_from_parquet(path: Path, descriptions: dict) -> list[dict]:
    """Column names/types from the parquet schema (no data is read), with the descriptions from the metadata."""
    schema = pq.read_schema(path)
    return [
        {"name": name, "type": str(dtype), "description": descriptions.get(name, "")}
        for name, dtype in zip(schema.names, schema.types)
    ]


def create_collection(metadata: dict, collection_id: str) -> pystac.Collection:
    providers = [
        pystac.Provider(
            name=metadata["PROVIDERS"]["name"],
            roles=[pystac.provider.ProviderRole.PRODUCER, pystac.provider.ProviderRole.LICENSOR],
            url=metadata["PROVIDERS"]["url"],
        ),
        pystac.Provider(
            name="Deltares",
            roles=[pystac.provider.ProviderRole.PROCESSOR, pystac.provider.ProviderRole.HOST],
            url="https://deltares.nl",
        ),
    ]

    extent = pystac.Extent(
        pystac.SpatialExtent([metadata["SPATIAL_EXTENT"]]),
        pystac.TemporalExtent([[start_datetime(metadata), None]]),
    )

    collection = pystac.Collection(
        id=collection_id,
        title=metadata["TITLE"],
        description=metadata["DESCRIPTION"],
        license=parse_license(metadata),
        providers=providers,
        extent=extent,
        catalog_type=pystac.CatalogType.RELATIVE_PUBLISHED,
    )
    collection.keywords = metadata["KEYWORDS"]

    pystac.extensions.item_assets.ItemAssetsExtension.add_to(collection)
    collection.extra_fields["item_assets"] = {
        "data": {
            "type": PARQUET_MEDIA_TYPE,
            "title": metadata["TITLE"],
            "roles": ["data"],
            "description": metadata["SHORT_DESCRIPTION"],
        }
    }

    pystac.extensions.scientific.ScientificExtension.add_to(collection)
    collection.extra_fields["sci:citation"] = metadata["CITATION"]
    if metadata.get("DOI"):
        collection.extra_fields["sci:doi"] = metadata["DOI"].removeprefix("https://doi.org/")
    return collection


def create_item(parquet_path: Path, item_id: str, metadata: dict, storage_prefix: str) -> pystac.Item:
    bbox = list(metadata["SPATIAL_EXTENT"])
    geometry = shapely.geometry.mapping(shapely.geometry.box(*bbox))

    n_rows = pq.ParquetFile(parquet_path).metadata.num_rows
    item = pystac.Item(
        id=item_id,
        geometry=geometry,
        bbox=bbox,
        datetime=start_datetime(metadata),
        properties={
            "title": metadata["TITLE_ABBREVIATION"],
            "description": metadata["SHORT_DESCRIPTION"],
            "table:row_count": int(n_rows),
            "table:columns": table_columns_from_parquet(parquet_path, metadata["TABLE_COLUMNS"]),
            "proj:epsg": int(metadata["CRS"].split(":")[1]),
            "deltares:item_key": item_id,
        },
        stac_extensions=[PROJ_SCHEMA_URI, TABLE_SCHEMA_URI],
    )
    item.common_metadata.created = datetime.datetime.now(datetime.timezone.utc)

    asset = pystac.Asset(
        href=urljoin(storage_prefix, parquet_path.name),
        media_type=PARQUET_MEDIA_TYPE,
        title=metadata["TITLE_ABBREVIATION"],
        description=metadata["SHORT_DESCRIPTION"],
        roles=["data"],
    )
    asset.extra_fields["file:size"] = os.path.getsize(parquet_path)
    item.add_asset("data", asset)
    return item

## 3) Create the STAC collection

In [4]:
collection = create_collection(metadata, collection_id)

print(f"Collection ready: {collection.id}")
print(f"Title: {metadata['TITLE']}")
print(f"Provider: {metadata['PROVIDERS']['name']}")
print(f"License: {collection.license}")
print(f"Spatial extent: {metadata['SPATIAL_EXTENT']}")
print(f"Temporal extent: {metadata['TEMPORAL_EXTENT']}")

Collection ready: country_boundaries
Title: Admin 0 – Countries
Provider: Natural Earth
License: CC-BY-4.0
Spatial extent: [-180.0, -90.0, 180.0, 83.634101]
Temporal extent: ['2022-01-01T00:00:00Z', '']


## 4) Build the STAC item

One item for the single GeoParquet. The columns come from the parquet schema.

In [5]:
item = create_item(parquet_path, collection_id, metadata, storage_prefix=href_prefix)
collection.add_item(item)

print(f"Item id: {item.id}")
print(f"Item bbox: {item.bbox}")
print(f"Data asset href: {item.assets['data'].href}")
print(f"table:row_count: {item.properties['table:row_count']}")
print(f"table:columns: {len(item.properties['table:columns'])} columns")

Item id: country_boundaries
Item bbox: [-180.0, -90.0, 180.0, 83.634101]
Data asset href: https://storage.googleapis.com/gca-data-public/gca/country_boundaries/country_boundaries.parquet
table:row_count: 258
table:columns: 170 columns


## 5) Save STAC catalog locally

Writes `STAC/data/current/country_boundaries/` and adds the collection to the root `catalog.json`.

In [6]:
stac_io = DefaultStacIO()
layout = pystac.layout.BestPracticesLayoutStrategy()

(stac_dir / collection_id / item.id).mkdir(parents=True, exist_ok=True)

collection.update_extent_from_items()

catalog = pystac.Catalog.from_file(str(stac_dir / "catalog.json"))
if catalog.get_child(collection.id):
    catalog.remove_child(collection.id)
    print(f"Removed existing child: {collection.id}")

catalog.add_child(collection)
collection.normalize_hrefs(str(stac_dir / collection_id), strategy=layout)
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED, dest_href=str(stac_dir), stac_io=stac_io)

collection.validate_all()
catalog.validate_all()

expected = [stac_dir / collection_id / "collection.json", stac_dir / collection_id / item.id / f"{item.id}.json"]
missing = [str(p) for p in expected if not p.is_file()]
if missing:
    raise FileNotFoundError(f"STAC outputs incomplete after save: {missing}")
print(f"STAC saved to: {stac_dir}")

STAC saved to: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current


## 6) Upload GeoParquet to Google Cloud (optional)

Uploads `country_boundaries.parquet` to `gs://gca-data-public/gca/country_boundaries/`, matching the href in the STAC item. An existing remote copy is replaced.

In [7]:
if RUN_UPLOAD:
    import gcsfs

    load_google_credentials(google_token_fp=google_cred_path)
    fs = gcsfs.GCSFileSystem(gcs_project, token=os.environ["GOOGLE_APPLICATION_CREDENTIALS"])

    target = urljoin(bucket_name, bucket_proj, proj_name, parquet_path.name)
    print(f"Uploading {parquet_path.name} ({parquet_path.stat().st_size / 1e6:.1f} MB) -> gs://{target} ...")
    if fs.exists(target):
        print("  Removing existing remote file")
        fs.rm(target)
    fs.put(str(parquet_path), target)
    print("Done")
else:
    print("RUN_UPLOAD is False, skipping GeoParquet upload.")

C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:339: FutureWarning: This function will be deprecated in the future, please use environment variables instead. When Google cloud is installed on your computer credentials can set using 'GOOGLE_DEFAULT' in the storage_kwargs argument
  warnings.warn(
C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:352: CredentialLeakageWarning: Keys loaded from shared network drive.
  warnings.warn(


Google Application Credentials load into environment.
Uploading country_boundaries.parquet (5.9 MB) -> gs://gca-data-public/gca/country_boundaries/country_boundaries.parquet ...
Done


## 7) Upload STAC catalog to Google Cloud (optional)

Replaces only this collection plus the root `catalog.json` at `gs://gca-data-public/gca/gca-stac-7/`.

In [8]:
if RUN_UPLOAD:
    local_collection_dir = stac_dir / collection_id
    remote_collection = urljoin(bucket_name, bucket_proj, stac_cloud_name, collection_id)
    remote_catalog = urljoin(bucket_name, bucket_proj, stac_cloud_name, "catalog.json")

    if fs.exists(remote_collection):
        print("Removing existing remote collection...")
        fs.rm(remote_collection, recursive=True)

    print("Uploading collection...")
    fs.put(str(local_collection_dir), remote_collection, recursive=True)
    print("Uploading root catalog.json...")
    fs.put(str(stac_dir / "catalog.json"), remote_catalog)

    print(f"Uploaded collection: gs://{remote_collection}/")
    print(f"Uploaded catalog: gs://{remote_catalog}")
else:
    print("RUN_UPLOAD is False, skipping STAC catalog upload.")

Uploading collection...
Uploading root catalog.json...
Uploaded collection: gs://gca-data-public/gca/gca-stac-7/country_boundaries/
Uploaded catalog: gs://gca-data-public/gca/gca-stac-7/catalog.json


## 8) Verify the cloud copy

The public HTTPS href must be reachable and have the same size and row count as the local GeoParquet.

In [9]:
import fsspec

cloud_href = item.assets["data"].href
resp = requests.head(cloud_href, timeout=60, allow_redirects=True)
resp.raise_for_status()
cloud_size = int(resp.headers["Content-Length"])
local_size = parquet_path.stat().st_size
print(f"cloud: {cloud_size} bytes | local: {local_size} bytes")
assert cloud_size == local_size, "Cloud and local file sizes differ"

with fsspec.open(cloud_href).open() as f:
    cloud_rows = pq.ParquetFile(f).metadata.num_rows
print(f"cloud rows: {cloud_rows} | local rows: {item.properties['table:row_count']}")
assert cloud_rows == item.properties["table:row_count"]
print("OK: the cloud GeoParquet matches the local file.")

cloud: 5945458 bytes | local: 5945458 bytes
cloud rows: 258 | local rows: 258
OK: the cloud GeoParquet matches the local file.
